# Ternary QAT for a 36M Mixtral MoE

This notebook trains the `ggml-org/stories15M_MOE` model with ternary
quantization-aware training. It is the small testbed behind the 21.78 PPL
recipe: learnable group scales, 0.1% outlier protection, p99.9 calibration and
layers 0 and 5 kept in FP32.

**Runtime:** about 10-20 minutes on a free Colab T4. The model is only 36M
parameters, so no paid GPU is needed.

**What you get:** `runs/best/qat_best.pt`, `metrics.json` and `history.csv`.


In [ ]:
!git clone https://github.com/ggarembroo/stories15m-moe-qat.git
%cd stories15m-moe-qat
!pip install -q -r requirements.txt

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Smoke test

Twenty steps on a small validation slice. This checks the model download, the
QAT module and the training loop before the full run.

In [ ]:
!python train_qat.py --preset best --steps 20 --batch-size 2 --max-val-chunks 10 --output-dir runs/smoke

## 2. Full recipe

600 steps, batch 4, sequence length 128. On a T4 this is typically 10-20
minutes. On CPU it is slower but still runs.

In [ ]:
!python train_qat.py --preset best --steps 600 --batch-size 4 --output-dir runs/best

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

metrics = json.load(open("runs/best/metrics.json"))
print(json.dumps(metrics, indent=2))

history = pd.read_csv("runs/best/history.csv")
plt.figure(figsize=(8, 3))
plt.plot(history["step"], history["loss"], linewidth=1)
plt.xlabel("step")
plt.ylabel("training loss")
plt.title("Ternary QAT on stories15M_MOE")
plt.grid(alpha=0.3)
plt.show()

## What to expect

| Variant | PPL |
|---|---:|
| FP32 | 16.20 |
| Ternary PTQ | 3404.56 |
| Ternary QAT, all params | 24.19 |
| + outlier + calibration + mixed precision | **21.78 best / 21.88 reproduced** |

The exact numbers depend on the GPU, seed and library versions. The point is the
gap between PTQ and QAT, not the third decimal.

## Notes

- The script downloads `TinyStories-valid.txt` from Hugging Face on first run.
- `runs/best/metrics.json` contains FP32, PTQ and post-QAT PPL.
- `runs/best/history.csv` contains step, loss and elapsed seconds.
- To keep the checkpoint in Drive, mount Drive and copy `runs/best` there.
